# As estatísticas da aula 2 em SQL

## Task 2: traduzir as estatísticas da aula 2 para SQL e guardar a consulta no projeto

As estatísticas da aula 2, agora em SQL.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-estatisticas-sql")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o código do vídeo 3.1.

In [2]:
# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")

## Gols por Copa em SQL

O `CASE WHEN` é o `F.when` do 2.3: os empates contados com `SUM`.

In [7]:
spark.sql("""
    SELECT
        ano,
        COUNT(*) AS partidas,
        SUM(total_gols) AS gols,
        ROUND(AVG(total_gols), 2) AS media_gols,
        SUM(CASE WHEN resultado = 'empate' THEN 1 ELSE 0 END) AS empates,
        ROUND(AVG(publico), 0) AS publico_medio
    FROM partidas
    GROUP BY ano
    ORDER BY ano
""").show(5)

+----+--------+----+----------+-------+-------------+
| ano|partidas|gols|media_gols|empates|publico_medio|
+----+--------+----+----------+-------+-------------+
|1930|      18|  70|      3.89|      0|      32808.0|
|1934|      17|  70|      4.12|      1|      21353.0|
|1938|      18|  84|      4.67|      3|      20872.0|
|1950|      22|  88|       4.0|      3|      47511.0|
|1954|      26| 140|      5.38|      2|      29562.0|
+----+--------+----+----------+-------+-------------+
only showing top 5 rows


## A tabela por seleção: CTE e UNION ALL

- **CTE** (`WITH nome AS (...)`): uma consulta intermediária com nome.
- **`UNION ALL`**: empilha os resultados de duas consultas.

In [8]:
spark.sql("""

WITH linhas_por_selecao AS (
    SELECT 
        ano, 
        mandante AS selecao,
        gols_mandante AS gols_pro,
        gols_visitante AS gols_contra
    FROM partidas
    UNION ALL
    SELECT 
        ano, 
        visitante AS selecao,
        gols_visitante AS gols_pro,
        gols_mandante AS gols_contra
    FROM partidas
)

SELECT ano,
       selecao,
       COUNT(*)                                                AS jogos,
       SUM(CASE WHEN gols_pro > gols_contra THEN 1 ELSE 0 END) AS vitorias,
       SUM(CASE WHEN gols_pro = gols_contra THEN 1 ELSE 0 END) AS empates,
       SUM(CASE WHEN gols_pro < gols_contra THEN 1 ELSE 0 END) AS derrotas,
       SUM(gols_pro)                                           AS gols_pro,
       SUM(gols_contra)                                        AS gols_contra,
       SUM(gols_pro) - SUM(gols_contra)                        AS saldo
FROM linhas_por_selecao
GROUP BY ano, selecao

""").show(5)

+----+-----------+-----+--------+-------+--------+--------+-----------+-----+
| ano|    selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+-----------+-----+--------+-------+--------+--------+-----------+-----+
|2002|   Portugal|    3|       1|      0|       2|       6|          4|    2|
|2010|    Uruguay|    7|       3|      2|       2|      11|          8|    3|
|1994|     Norway|    3|       1|      1|       1|       1|          1|    0|
|1954|Switzerland|    4|       2|      0|       2|      11|         11|    0|
|1998|    Romania|    4|       2|      1|       1|       4|          3|    1|
+----+-----------+-----+--------+-------+--------+--------+-----------+-----+
only showing top 5 rows


## Guardando a consulta no projeto

Consultas reaproveitadas ficam em arquivos `.sql`, versionados com o código.

In [9]:
from pathlib import Path
Path("../../sql").mkdir(exist_ok=True)

In [10]:
%%writefile ../../sql/gols_por_copa.sql
SELECT
    ano,
    COUNT(*) AS partidas,
    SUM(total_gols) AS gols,
    ROUND(AVG(total_gols), 2) AS media_gols,
    SUM(CASE WHEN resultado = 'empate' THEN 1 ELSE 0 END) AS empates,
    ROUND(AVG(publico), 0) AS publico_medio
FROM partidas
GROUP BY ano
ORDER BY ano

Writing ../../sql/gols_por_copa.sql


In [11]:
%%writefile ../../sql/gols_por_selecao.sql
WITH linhas_por_selecao AS (
    SELECT 
        ano, 
        mandante AS selecao,
        gols_mandante AS gols_pro,
        gols_visitante AS gols_contra
    FROM partidas
    UNION ALL
    SELECT 
        ano, 
        visitante AS selecao,
        gols_visitante AS gols_pro,
        gols_mandante AS gols_contra
    FROM partidas
)

SELECT ano,
       selecao,
       COUNT(*)                                                AS jogos,
       SUM(CASE WHEN gols_pro > gols_contra THEN 1 ELSE 0 END) AS vitorias,
       SUM(CASE WHEN gols_pro = gols_contra THEN 1 ELSE 0 END) AS empates,
       SUM(CASE WHEN gols_pro < gols_contra THEN 1 ELSE 0 END) AS derrotas,
       SUM(gols_pro)                                           AS gols_pro,
       SUM(gols_contra)                                        AS gols_contra,
       SUM(gols_pro) - SUM(gols_contra)                        AS saldo
FROM linhas_por_selecao
GROUP BY ano, selecao

Writing ../../sql/gols_por_selecao.sql


In [12]:
consulta = Path("../../sql/gols_por_copa.sql").read_text(encoding="utf-8")
spark.sql(consulta).show(3)

+----+--------+----+----------+-------+-------------+
| ano|partidas|gols|media_gols|empates|publico_medio|
+----+--------+----+----------+-------+-------------+
|1930|      18|  70|      3.89|      0|      32808.0|
|1934|      17|  70|      4.12|      1|      21353.0|
|1938|      18|  84|      4.67|      3|      20872.0|
+----+--------+----+----------+-------+-------------+
only showing top 3 rows


## SQL ou DataFrame API?

| Use SQL quando... | Use a DataFrame API quando... |
|---|---|
| a consulta é declarativa e longa (vários joins, CTEs) | há lógica de programa (laços, parâmetros, condições) |
| quem vai ler ou manter conhece SQL | você quer reaproveitar a transformação como função |
| é uma consulta de análise, para explorar | a transformação vai entrar nas tabelas do projeto |

## Resultado da task 2

As estatísticas da aula 2 em SQL, e a consulta em `sql/gols_por_copa.sql`.

## Para consultar depois

- **A armadilha do `NOT IN`:** `WHERE x NOT IN (SELECT ...)` devolve **vazio** se a subconsulta tiver um único `NULL`,
  porque, para o SQL, comparar com `NULL` não dá verdadeiro nem falso. Para "o que está em A e não está em B", prefira
  `LEFT ANTI JOIN` ou `NOT EXISTS`.
- **Joins em SQL** seguem a sintaxe que você conhece: `FROM partidas p JOIN copas c ON p.ano = c.ano`.
- **`UNION` e `UNION ALL`:** o `UNION` remove as linhas duplicadas, e o `UNION ALL` mantém todas. Na tabela por
  seleção, o `UNION` juntaria dois placares iguais da mesma seleção na mesma Copa, e os jogos seriam contados a menos.
  No DataFrame, `union` e `unionByName` se comportam como `UNION ALL`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [13]:
spark.stop()